# Eclat, Groceries và thực nghiệm benchmark

Notebook dùng **kernel Julia** để chuẩn bị dữ liệu, kiểm chứng Eclat Set/BitVector với SPMF, phân tích Groceries và chạy bốn nhóm thực nghiệm. Fashion-MNIST có ba notebook riêng trong README.

Trong Colab, chọn **Runtime → Change runtime type → Julia**, runtime CPU, rồi chạy các ô theo thứ tự. Nếu tài khoản chưa có kernel Julia, mở notebook bằng Jupyter/IJulia trên máy có Julia; không chạy các ô Julia bằng kernel Python.

Mặc định mỗi cấu hình đo một lượt để kiểm tra toàn bộ luồng. Đặt `BENCHMARK_RUNS = 5` để lấy trung vị năm lượt như thiết kế báo cáo. Số đo mới thuộc runtime hiện tại; không tự thay thế số liệu đã công bố. Dữ liệu và công cụ được tải công khai, không cần token. Kết quả trong `/content` mất khi runtime kết thúc: tải gói kết quả ở ô cuối.


## 1. Môi trường và cấu hình

Source được ghim tại commit công khai `c93fed0c436275e51f8672f6e7e8182d9b855040`, chứa đủ các script thực nghiệm. Notebook tạo checkout riêng, không đổi nhánh hay ghi đè kết quả trong repo đang làm việc. Các file benchmark được tải trực tiếp từ bản Hugging Face đã xác minh; bước chuẩn hóa tái sử dụng script của nhóm.

Đặt số luồng đo bằng một. Các worker benchmark chạy trong tiến trình riêng; thay đổi biến môi trường ở ô này không thay đổi số luồng của kernel Julia đã khởi động.


In [ ]:
using Pkg, Downloads, SHA, TOML, Dates, LibGit2, Markdown

BENCHMARK_RUNS = 1  # 5: tái lập thiết kế năm lượt/cấu hình trong báo cáo
BENCHMARK_RUNS > 0 || error("BENCHMARK_RUNS phải dương")
SOURCE_COMMIT = "c93fed0c436275e51f8672f6e7e8182d9b855040"
REPOSITORY_URL = "https://github.com/NguyenVoTanDuy2006/DataMining-FIM-Project.git"
ENV["JULIA_NUM_THREADS"] = "1"
ENV["OPENBLAS_NUM_THREADS"] = "1"
ENV["MPLBACKEND"] = "Agg"
ENV["JULIA_PKG_PRECOMPILE_AUTO"] = "0"

work_root = isdir("/content") ? "/content" : tempdir()
REPO_ROOT = joinpath(work_root, "eclat-demo-" * first(SOURCE_COMMIT, 8))
if ispath(REPO_ROOT) && !isdir(joinpath(REPO_ROOT, ".git"))
    error("Checkout chưa hoàn tất tại $REPO_ROOT. Khởi động lại runtime hoặc chọn thư mục làm việc mới; notebook giữ nguyên thư mục hiện có.")
end
repo = isdir(joinpath(REPO_ROOT, ".git")) ? LibGit2.GitRepo(REPO_ROOT) :
    LibGit2.clone(REPOSITORY_URL, REPO_ROOT)
try
    LibGit2.checkout!(repo, SOURCE_COMMIT; force=true)
    string(LibGit2.head_oid(repo)) == SOURCE_COMMIT || error("Source commit không khớp")
finally
    close(repo)
end
cd(REPO_ROOT)
Pkg.activate(REPO_ROOT)
Pkg.instantiate()
using EclatFIM
using EclatFIM.FashionMNIST.FashionTransactionIO: csv_fields

mkpath(joinpath(REPO_ROOT, "results", "colab"))
RUN_ROOT = mktempdir(joinpath(REPO_ROOT, "results", "colab"); prefix="eclat_")
JULIA = Base.julia_cmd()
TESTS_PASSED = false
run_julia(script, args...) = run(`$JULIA --startup-file=no --color=no --threads=1 --project=$REPO_ROOT $script $args`)
function show_csv(path)
    rows = csv_fields.(readlines(path))
    isempty(rows) && error("CSV rỗng: $path")
    escape_field(value) = replace(value, "|" => "\\|", "_" => "\\_", "\n" => " ")
    table = ["| " * join(escape_field.(row), " | ") * " |" for row in rows]
    insert!(table, 2, "| " * join(fill("---", length(rows[1])), " | ") * " |")
    display(Markdown.parse(join(table, "\n")))
end
println("Julia: ", VERSION, "; kernel threads: ", Threads.nthreads())
println("Source: ", SOURCE_COMMIT, "; thư mục kết quả: ", RUN_ROOT)


## 2. Tải và chuyển đổi dữ liệu

Bốn benchmark từ [Hugging Face](https://huggingface.co/datasets/Yelgsef/fimi-eclat-benchmarks) được ghim phiên bản, kiểm tra SHA-256 và giữ nguyên giao dịch trùng. Mushroom là bản **8.124 giao dịch**. Retail/T10I4D100K cộng một vào ID để thỏa điều kiện item dương; Julia và SPMF nhận cùng file chuẩn hóa.

Groceries tải từ nguồn CSV công khai, chuẩn hóa tên sản phẩm, bỏ item lặp trong từng giỏ và giữ các giỏ trùng. `prepare_data.jl` cũng chạy pipeline Groceries: minsup 1%, minconf 50% và bảng 12 cấu hình.


In [ ]:
include(joinpath(REPO_ROOT, "scripts", "prepare_data.jl"))
BENCHMARK_URL = "https://huggingface.co/datasets/Yelgsef/fimi-eclat-benchmarks/resolve/92ce74f1107250b7c8cd79df07ba415df58ca98d"
for (name, digest) in PrepareData.CHECKSUMS
    path = joinpath(REPO_ROOT, "data", "benchmark", "raw", "$name.dat")
    PrepareData.verified_download("$BENCHMARK_URL/$name.dat", path, digest)
    println("SHA-256 PASS: ", name)
end
PrepareData.main(String[])


## 3. Java và SPMF tham chiếu

SPMF chỉ dùng để kiểm chứng và so sánh, không thay thế Eclat của nhóm. Nếu Java hiện tại dưới 25, ô này tải Temurin JRE 25.0.2 vào thư mục tạm và xác minh SHA-256. JAR SPMF được ghim từ bản công khai trên Hugging Face, kèm checksum để tránh thay phiên bản giữa các lần chạy. Xem [SPMF chính thức](https://www.philippe-fournier-viger.com/spmf/) để biết tài liệu thuật toán.


In [ ]:
java = Sys.which("java")
java_version = isnothing(java) ? "" : try read(`$java --version`, String) catch; "" end
version_match = match(r"(?:openjdk|java) (\d+)", java_version)
if isnothing(version_match) || parse(Int, version_match[1]) < 25
    Sys.islinux() && Sys.ARCH == :x86_64 || error("Tự cài JRE hỗ trợ Linux x86_64; hãy cài Java 25 cho hệ hiện tại")
    runtime_root = joinpath(work_root, "eclat-temurin-25.0.2")
    mkpath(runtime_root)
    archive = joinpath(runtime_root, "jre.tar.gz")
    url = "https://github.com/adoptium/temurin25-binaries/releases/download/jdk-25.0.2%2B10/OpenJDK25U-jre_x64_linux_hotspot_25.0.2_10.tar.gz"
    PrepareData.verified_download(url, archive,
        "d6c89e08f42be94cd55eab20190958a35b993625018a3ac59cb3d16d8445cf98")
    candidates = filter(p -> isfile(joinpath(p, "bin", "java")), readdir(runtime_root; join=true))
    if isempty(candidates)
        run(`tar -xzf $archive -C $runtime_root`)
        candidates = filter(p -> isfile(joinpath(p, "bin", "java")), readdir(runtime_root; join=true))
    end
    length(candidates) == 1 || error("Không tìm thấy đúng một JRE")
    ENV["PATH"] = joinpath(only(candidates), "bin") * ":" * ENV["PATH"]
end
JAVA = something(Sys.which("java"))
run(`$JAVA --version`)
SPMF_JAR = joinpath(REPO_ROOT, "data", "benchmark", "tools", "spmf.jar")
PrepareData.verified_download(
    "https://huggingface.co/datasets/TheBrilliantCoder/frequent-itemset-mining-data/resolve/263201a1ca17c39eb0906dbb094b4838f795fe8f/tools/spmf.jar",
    SPMF_JAR, "2cc2d89e39909230ea0455da4a1c79b389dc98ca04b0670befb63e679b5e2ea0")
println("SPMF SHA-256: ", bytes2hex(open(sha256, SPMF_JAR)))


## 4. Ví dụ Eclat và giao diện dòng lệnh

Với năm giao dịch dưới đây và minsup 40%, ngưỡng đếm là hai. Cả hai phiên bản phải trả về cùng itemsets/support. Một dòng `1 2 3 #SUP: 2` nghĩa là tập `{1,2,3}` xuất hiện trong hai giao dịch.


In [ ]:
transactions = [[1,2,3], [1,3], [2,3], [1,2], [1,2,3]]
set_result = mine(transactions, 0.4; backend=:set)
bit_result = mine(transactions, 0.4; backend=:bitvector)
@assert set_result == bit_result
@assert set_result[(1,2,3)] == 2
write_spmf_itemsets(stdout, set_result)
toy_input = joinpath(RUN_ROOT, "toy.txt")
write(toy_input, join((join(t, " ") for t in transactions), "\n") * "\n")
toy_output = joinpath(RUN_ROOT, "toy_itemsets.txt")
run_julia("scripts/eclat_cli.jl", "--input", toy_input, "--minsup", "0.4",
    "--backend", "bitvector", "--output", toy_output)
@assert read_spmf_itemsets(toy_output) == set_result


## 5. Tạo đáp án SPMF và chạy toàn bộ test

Các đáp án được **tạo từ SPMF trước khi chạy test**, không lấy output Julia làm tham chiếu. Năm CSDL được kiểm tra toàn bộ ở các ngưỡng: Groceries 1%, Chess 87,5%, Mushroom 70%, Retail 4%, T10I4D100K 1,5%. Test so sánh từng itemset và số hỗ trợ cho cả Set và BitVector; thiếu file hoặc sai kết quả sẽ dừng notebook.


In [ ]:
TESTS_PASSED = false
reference_cases = [
    ("Groceries", "data/processed/groceries_spmf.txt", "results/groceries/groceries_spmf_0.01.txt", "1%", 333),
    ("Chess", "data/benchmark/processed/processes/chess_spmf.txt", "results/benchmarks/peak_ram/spmf/chess_0.875.txt", "87.5%", 1386),
    ("Mushroom", "data/benchmark/processed/processes/mushroom_spmf.txt", "results/benchmarks/peak_ram/spmf/mushroom_0.7.txt", "70%", 31),
    ("Retail", "data/benchmark/processed/processes/retail_spmf.txt", "results/benchmarks/peak_ram/spmf/retail_0.04.txt", "4%", 18),
    ("T10I4D100K", "data/benchmark/processed/processes/T10I4D100K_spmf.txt", "results/benchmarks/peak_ram/spmf/T10I4D100K_0.015.txt", "1.5%", 237),
]
reference_root = joinpath(RUN_ROOT, "spmf_references")
mkpath(reference_root)
for (name, input, output, minsup, expected_count) in reference_cases
    mkpath(dirname(output))
    temporary = output * ".tmp"
    try
        run(`$JAVA -jar $SPMF_JAR run Eclat $input $temporary $minsup`)
        payload = filter(line -> !startswith(strip(line), "@"), readlines(temporary))
        @assert length(read_spmf_itemsets(IOBuffer(join(payload, "\n")))) == expected_count
        mv(temporary, output; force=true)
    finally
        isfile(temporary) && rm(temporary)
    end
    cp(output, joinpath(reference_root, "$name.txt"); force=true)
end
test_log = joinpath(RUN_ROOT, "tests.txt")
try
    open(test_log, "w") do io
        run(pipeline(`$JULIA --startup-file=no --color=no --threads=1 --project=$REPO_ROOT test/runtests.jl`;
            stdout=io, stderr=io))
    end
    TESTS_PASSED = true
finally
    print(read(test_log, String))
end


## 6. Groceries: từ giỏ hàng đến luật kết hợp

Bảng đầu cho biết số itemsets và luật theo 12 cặp ngưỡng. Bảng sau là mười luật có lift cao nhất ở minsup 1%/minconf 50%.

Ví dụ, luật `citrus fruit + root vegetables → other vegetables` có 102 giỏ hỗ trợ, confidence khoảng 58,62% và lift 3,03. Đây là liên hệ đồng xuất hiện để khảo sát gợi ý mua kèm; chưa chứng minh gợi ý làm tăng doanh thu. Đọc lift cùng số giỏ hỗ trợ, tránh diễn giải luật ít gặp chỉ từ lift cao.


In [ ]:
show_csv("results/groceries/parameter_summary.csv")
show_csv("results/groceries/top10_rules.csv")
cp("results/groceries", joinpath(RUN_ROOT, "groceries"); force=true)


## 7. Benchmark và biểu đồ

Bốn ô tiếp theo đo toàn bộ bốn CSDL × bảy minsup, Peak RSS, các tập con Retail 10/25/50/75/100% và độ dài giao dịch tổng hợp 4/8/12/16/20. `BENCHMARK_RUNS = 1` là kiểm tra luồng, không đủ để mô tả độ biến thiên; dùng năm lượt cho thiết kế báo cáo.

- Runtime toàn tiến trình gồm khởi động/JIT/đọc ghi.
- Peak RSS là đỉnh bộ nhớ toàn tiến trình, không phải riêng cấu trúc TID-set.
- Scalability và độ dài giao dịch đo `mine` sau làm nóng trên toàn bộ input.

Điều phối, chuẩn hóa và khai thác đều bằng Julia. Các script Python/matplotlib có sẵn chỉ đọc CSV và vẽ biểu đồ; không khai thác itemsets. Ô sau chỉ cài matplotlib nếu môi trường chưa có.


In [ ]:
PYTHON = something(Sys.which("python3"), Sys.which("python"))
if !success(`$PYTHON -c "import matplotlib"`)
    plot_environment = joinpath(work_root, "eclat-plot-python")
    plot_python = joinpath(plot_environment, "bin", "python")
    isfile(plot_python) || run(`$PYTHON -m venv $plot_environment`)
    PYTHON = plot_python
    success(`$PYTHON -c "import matplotlib"`) || run(`$PYTHON -m pip install matplotlib`)
end
function benchmark_and_show(name, plot_script)
    destination = joinpath(RUN_ROOT, "benchmarks", name)
    mining_complete = joinpath(destination, "mining_complete")
    if ispath(destination)
        isfile(mining_complete) || error("Kết quả $name chưa đo xong. Chạy lại từ ô cấu hình, rồi chạy các ô tiếp theo để tạo run mới.")
    else
        run_julia("scripts/benchmark_$name.jl", "--runs", string(BENCHMARK_RUNS),
            "--output-dir", destination)
        touch(mining_complete)
    end
    run(`$PYTHON $plot_script $destination`)
    show_csv(joinpath(destination, "summary.csv"))
    for path in sort(readdir(destination; join=true))
        endswith(path, ".png") && display(MIME"image/png"(), read(path))
    end
end


### 7.1. Thời gian và số itemsets theo minsup

Đối chiếu Set, BitVector và SPMF trên cùng dữ liệu/ngưỡng. Mỗi output được kiểm tra toàn bộ itemsets/support trước khi tổng hợp.


In [ ]:
benchmark_and_show("processes", "scripts/plot_process_benchmarks.py")


### 7.2. Peak RSS tại ngưỡng giữa dải

So sánh Set và BitVector trên bốn CSDL. Số liệu có cả môi trường thực thi, JIT và input; không lấy hiệu hai mức đỉnh để suy ra RAM riêng thuật toán.


In [ ]:
benchmark_and_show("peak_ram", "scripts/plot_peak_ram.py")


### 7.3. Khả năng mở rộng trên Retail

Giữ minsup tương đối 4%, seed 20261116, lấy các tập con lồng nhau. Ngưỡng support count thay đổi theo số giao dịch.


In [ ]:
benchmark_and_show("scalability", "scripts/plot_scalability.py")


### 7.4. Ảnh hưởng độ dài giao dịch

Giữ N=5.000, M=64, minsup 5%, seed 20261116. Các giỏ dài hơn mở rộng từ cùng hoán vị item của từng giao dịch; số kết quả và support được đối chiếu với SPMF.


In [ ]:
benchmark_and_show("transaction_length", "scripts/plot_transaction_length.py")


## 8. Lưu môi trường và tải kết quả

Chỉ đóng gói khi test và bốn nhóm thực nghiệm hoàn tất. Gói chứa bảng Groceries, tham chiếu SPMF, log test, CSV/biểu đồ benchmark và thông tin nguồn/môi trường. Trong Colab, archive nằm ngay trong `/content`. Mở **Files**, tìm tên archive được in bên dưới, chọn **Download** trước khi đóng runtime. Nếu vẽ biểu đồ lỗi sau khi đo xong, sửa lỗi rồi chạy lại riêng ô benchmark đó; số đo đã hoàn tất được giữ lại.


In [ ]:
@assert TESTS_PASSED "Chưa có lượt test thành công"
for relative in ("tests.txt", "groceries/parameter_summary.csv", "groceries/top10_rules.csv",
        "groceries/groceries_spmf_0.01.txt", "spmf_references/Groceries.txt",
        "spmf_references/Chess.txt", "spmf_references/Mushroom.txt",
        "spmf_references/Retail.txt", "spmf_references/T10I4D100K.txt")
    path = joinpath(RUN_ROOT, relative)
    @assert isfile(path) && filesize(path) > 0 "Thiếu đầu ra $relative; chạy đủ các ô dữ liệu, test và Groceries"
end
for name in ("processes", "peak_ram", "scalability", "transaction_length")
    destination = joinpath(RUN_ROOT, "benchmarks", name)
    for filename in ("mining_complete", "summary.csv", "runs.csv")
        @assert isfile(joinpath(destination, filename)) "Thiếu kết quả $name/$filename"
    end
    stems = name == "processes" ? ("runtime_processes", "itemset_counts") : (name,)
    for stem in stems, extension in ("png", "svg")
        image = joinpath(destination, "$stem.$extension")
        @assert isfile(image) && filesize(image) > 0 "Thiếu biểu đồ $image; chạy lại ô benchmark tương ứng"
    end
end
cp("Project.toml", joinpath(RUN_ROOT, "Project.toml"); force=true)
cp("Manifest.toml", joinpath(RUN_ROOT, "Manifest.toml"); force=true)
metadata = Dict(
    "created_utc" => string(now(UTC)), "source_commit" => SOURCE_COMMIT,
    "julia_version" => string(VERSION), "kernel_threads" => Threads.nthreads(),
    "worker_threads" => 1, "benchmark_runs" => BENCHMARK_RUNS,
    "benchmark_revision" => "92ce74f1107250b7c8cd79df07ba415df58ca98d",
    "seed" => 20261116, "java_version" => read(`$JAVA --version`, String),
    "spmf_sha256" => bytes2hex(open(sha256, SPMF_JAR)),
    "cpu" => Sys.CPU_NAME, "cpu_threads" => Sys.CPU_THREADS,
    "ram_bytes" => Sys.total_memory(), "tests_passed" => TESTS_PASSED,
    "input_sha256" => Dict(name => bytes2hex(open(sha256, input))
        for (name, input, _, _, _) in reference_cases),
)
open(joinpath(RUN_ROOT, "environment.toml"), "w") do io
    TOML.print(io, metadata)
end
archive_path = joinpath(work_root, basename(RUN_ROOT) * ".tar.gz")
run(`tar -czf $archive_path -C $(dirname(RUN_ROOT)) $(basename(RUN_ROOT))`)
println("Hoàn tất. Tải kết quả tại: ", archive_path)
println("SHA-256 archive: ", bytes2hex(open(sha256, archive_path)))
